# Step 2.1 — Official Data Inventory and Source Contract

**Project:** Trace the Ace — Scratch Mastery Phase  
**Phase:** Data Foundation  
**Notebook:** `01_data_inventory.ipynb`  
**Notebook Version:** `data_inventory_v1`  
**Upstream Dependency:** `00_environment_and_paths.ipynb`  
**Downstream Consumer:** `02_turn_parser.ipynb`

---

## Notebook Mission

This notebook establishes a verified and reproducible contract for every official data source used by the Scratch Mastery pipeline.

It will:

1. verify the identity and availability of the official source files;
2. confirm the analytical unit and authoritative identifiers of each source;
3. inspect the physical and logical structure of the transcript population;
4. identify all observed transcript schema variants;
5. validate how transcripts map to tutoring sessions;
6. freeze a machine-readable source contract for the deterministic turn parser.

The notebook performs **source discovery and contract validation only**. It does not parse canonical turns, clean transcript text, engineer features, perform exploratory outcome analysis, or train a model.

## Purpose and Research-Engineering Questions

The Data Inventory stage must answer the following questions using direct evidence from the official files:

1. **Which official data sources are available?**
2. **What analytical unit does each source represent?**
3. **Which fields are the authoritative response, session, objective, label, and fold identifiers?**
4. **How are responses, labels, sessions, folds, and transcript files connected?**
5. **What is the actual physical format and internal structure of the transcript source?**
6. **Where are the utterance container, speaker role, text, timestamp, and utterance identifier stored?**
7. **Does one transcript structure describe the entire population, or are multiple structural variants present?**
8. **Can every transcript be assigned to a session using one deterministic and evidence-supported rule?**
9. **Are all parser-critical fields sufficiently confirmed to construct a reliable turn parser?**

### Primary Deliverable

The primary deliverable of this notebook is a validated, machine-readable:

`transcript_schema_contract.json`

This contract will be the only authoritative transcript-schema input used by `02_turn_parser.ipynb`.

## Scope, Exclusions, and Decision Language

### In Scope

This notebook may:

- read setup manifests produced by the environment notebook;
- verify official paths, file types, sizes, and fingerprints;
- inspect tabular schemas and identifier relationships;
- enumerate the complete transcript file population;
- inspect representative raw transcript structures;
- perform a lightweight structural scan across all transcript files;
- identify schema variants and parser-relevant exceptions;
- validate transcript-to-session mapping;
- generate inventory, issue, contract, and audit artefacts.

### Strictly Out of Scope

This notebook must not:

- create the canonical turn table;
- reorder transcript utterances;
- normalize or reassign speaker roles;
- clean, rewrite, summarize, or flatten transcript text;
- compare positive and negative outcome groups;
- calculate objective difficulty or label priors;
- perform semantic duplicate detection;
- create TF-IDF, embedding, retrieval, or model features;
- use baseline predictions or OOF probabilities;
- train, calibrate, or evaluate any predictive model.

---

## Evidence Decision Language

Every important schema conclusion must use one of the following evidence states:

| Evidence State | Meaning |
|---|---|
| `CONFIRMED` | Directly verified from the official source population |
| `INFERRED` | Suggested by evidence but not fully verified |
| `UNRESOLVED` | The available evidence does not support a reliable decision |

Every validation check must use one of the following execution states:

| Execution State | Meaning |
|---|---|
| `PASS` | The required condition is satisfied |
| `WARNING` | A documented exception exists but may have a safe handling policy |
| `FAIL` | A blocking requirement is not satisfied |

### Readiness Rule

`DATA_INVENTORY_READY` may be set to `True` only when every parser-critical contract field is `CONFIRMED` and every required validation check is `PASS`.

## Data Governance and Execution Principles

The following rules apply throughout this notebook.

### 1. Raw Sources Are Read-Only

Official source files must never be modified, overwritten, renamed, or cleaned in place.

### 2. Observation Must Be Separated from Policy

The notebook must distinguish between:

- what was observed in the raw data;
- what handling policy is proposed for the future parser.

Example:

- **Observed:** timestamps are absent in a subset of files;
- **Parser policy:** preserve original source order and flag missing timestamps.

### 3. No Silent Data Loss

No file, row, identifier, or structural variant may be silently removed or skipped. Every failure must be recorded with an explicit status and reason.

### 4. No Schema Guessing

Candidate field detection is diagnostic only. A candidate field must not become authoritative until it is validated against representative samples and the full transcript population.

### 5. Samples Discover; the Full Population Confirms

Representative samples may be used to discover candidate structures. Final schema decisions must be validated through a lightweight scan of the complete transcript population.

### 6. Raw and Canonical Concepts Must Remain Separate

Raw values such as role, timestamp, identifier, and text must remain distinguishable from any future canonical representation.

### 7. Label-Blind Transcript Discovery

Outcome labels may be inspected only to validate schema, key uniqueness, and binary-domain integrity. They must not influence transcript-schema, role, text, or parser-policy decisions.

### 8. Deterministic Outputs

The same official inputs and notebook version must produce the same:

- source inventory;
- schema decisions;
- relationship counts;
- issue counts;
- output hashes.

### 9. Explicit Audit Trail

Every final artefact must record its input identity, version, row or file count, schema, issue count, and reproducible hash.

### 10. Fail Before Publishing an Invalid Contract

Diagnostic artefacts may be saved after a failure, but an authoritative transcript contract must not be published while any parser-critical requirement remains unresolved.

In [1]:
from collections import Counter
from datetime import datetime
from pathlib import Path
from uuid import uuid4
import hashlib
import json
import os
import random
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
from IPython.display import display

NOTEBOOK_NAME = "01_data_inventory.ipynb"
NOTEBOOK_VERSION = "data_inventory_v1"
CONTRACT_VERSION = "transcript_schema_contract_v1"
RANDOM_SEED = 42
RUN_STARTED_AT = datetime.now().astimezone().isoformat()

EVIDENCE_STATES = ("CONFIRMED", "INFERRED", "UNRESOLVED")
EXECUTION_STATES = ("PASS", "WARNING", "FAIL")
BLOCKING_EXECUTION_STATE = "FAIL"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
pd.options.mode.chained_assignment = "raise"

warnings.simplefilter("default")

notebook_configuration = {
    "notebook_name": NOTEBOOK_NAME,
    "notebook_version": NOTEBOOK_VERSION,
    "contract_version": CONTRACT_VERSION,
    "run_started_at": RUN_STARTED_AT,
    "random_seed": RANDOM_SEED,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pyarrow_version": pa.__version__,
    "evidence_states": ", ".join(EVIDENCE_STATES),
    "execution_states": ", ".join(EXECUTION_STATES)
}

display(pd.DataFrame(notebook_configuration.items(), columns=["configuration", "value"]))

,configuration,value
0,notebook_name,01_data_inventory.ipynb
1,notebook_version,data_inventory_v1
2,contract_version,transcript_schema_contract_v1
3,run_started_at,2026-08-07T03:55:33.243904+06:00
4,random_seed,42
5,pandas_version,2.3.3
6,numpy_version,2.2.6
7,pyarrow_version,25.0.0
8,evidence_states,"CONFIRMED, INFERRED, UNRESOLVED"
9,execution_states,"PASS, WARNING, FAIL"


In [2]:
def _json_default(value):
    if value is pd.NA or value is None:
        return None
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, set):
        return sorted(value, key=str)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable.")


def atomic_save(target_path, writer):
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)

    temporary_name = f".{target_path.name}.{uuid4().hex}.tmp"
    temporary_path = target_path.parent / temporary_name

    try:
        writer(temporary_path)
        os.replace(temporary_path, target_path)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    return target_path


def read_json(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"JSON file was not found: {path}")

    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def write_json(payload, path):
    def writer(temporary_path):
        with temporary_path.open("w", encoding="utf-8") as file:
            json.dump(payload, file, indent=2, ensure_ascii=False, sort_keys=True, default=_json_default)

    return atomic_save(path, writer)


def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Hash target is not a readable file: {path}")

    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


def _normalise_hash_value(value):
    if value is None or value is pd.NA:
        return "<NA>"

    if isinstance(value, float) and np.isnan(value):
        return "<NA>"

    if isinstance(value, set):
        value = sorted(value, key=str)

    if isinstance(value, (dict, list, tuple, set, np.ndarray)):
        return json.dumps(value, ensure_ascii=False, sort_keys=True, default=_json_default)

    return value


def stable_table_hash(table, sort_by=None):
    if not isinstance(table, pd.DataFrame):
        raise TypeError("stable_table_hash expects a pandas DataFrame.")

    working = table.copy()

    if sort_by is not None:
        sort_columns = [sort_by] if isinstance(sort_by, str) else list(sort_by)
        missing_columns = [column for column in sort_columns if column not in working.columns]

        if missing_columns:
            raise KeyError(f"Hash sort columns are missing: {missing_columns}")

        working = working.sort_values(sort_columns, kind="mergesort", na_position="last")

    working = working.reset_index(drop=True)

    for column in working.select_dtypes(include="object").columns:
        working[column] = working[column].map(_normalise_hash_value)

    schema = [(str(column), str(working[column].dtype)) for column in working.columns]
    schema_bytes = json.dumps(schema, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    value_bytes = pd.util.hash_pandas_object(working, index=False, categorize=True).values.tobytes()

    digest = hashlib.sha256()
    digest.update(schema_bytes)
    digest.update(value_bytes)

    return digest.hexdigest()


def path_status(path):
    path = Path(path)

    status = {
        "path": str(path),
        "exists": path.exists(),
        "is_file": False,
        "is_directory": False,
        "suffix": path.suffix.lower(),
        "size_bytes": None,
        "modified_at": None,
        "accessible": False,
        "error": None
    }

    if not path.exists():
        return status

    try:
        file_stat = path.stat()
        status["is_file"] = path.is_file()
        status["is_directory"] = path.is_dir()
        status["size_bytes"] = int(file_stat.st_size) if path.is_file() else None
        status["modified_at"] = datetime.fromtimestamp(file_stat.st_mtime).astimezone().isoformat()
        status["accessible"] = os.access(path, os.R_OK)
    except OSError as error:
        status["error"] = f"{type(error).__name__}: {error}"

    return status


def safe_table_read(path, **kwargs):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Table file was not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        options = {"low_memory": False}
        options.update(kwargs)
        return pd.read_csv(path, **options)

    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path, **kwargs)

    if suffix == ".feather":
        return pd.read_feather(path, **kwargs)

    if suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True, **kwargs)

    raise ValueError(f"Unsupported tabular file format: {suffix}")


def compact_null_profile(table, source_name):
    if not isinstance(table, pd.DataFrame):
        raise TypeError("compact_null_profile expects a pandas DataFrame.")

    rows = []

    for column in table.columns:
        series = table[column]
        non_null = series.dropna()

        try:
            unique_count = int(series.nunique(dropna=True))
            unique_count_status = "AVAILABLE"
        except TypeError:
            unique_count = None
            unique_count_status = "UNHASHABLE_VALUES"

        rows.append({
            "source_name": source_name,
            "column_name": str(column),
            "dtype": str(series.dtype),
            "row_count": int(len(series)),
            "null_count": int(series.isna().sum()),
            "null_rate": float(series.isna().mean()),
            "unique_count": unique_count,
            "unique_count_status": unique_count_status,
            "example_python_type": type(non_null.iloc[0]).__name__ if len(non_null) else None
        })

    return pd.DataFrame(rows)


utility_registry = [
    "atomic_save",
    "read_json",
    "write_json",
    "sha256_file",
    "stable_table_hash",
    "path_status",
    "safe_table_read",
    "compact_null_profile"
]

print(f"Notebook utilities ready: {len(utility_registry)}")
display(pd.DataFrame({"utility": utility_registry, "status": "READY"}))

Notebook utilities ready: 8


,utility,status
0,atomic_save,READY
1,read_json,READY
2,write_json,READY
3,sha256_file,READY
4,stable_table_hash,READY
5,path_status,READY
6,safe_table_read,READY
7,compact_null_profile,READY


## Step 2.1.1 — Setup Entry Gate

Before inspecting any official source, this notebook must verify that the upstream environment-and-path setup completed successfully and produced the required reusable artefacts.

### Required Upstream Artefacts

The entry gate requires:

- `path_registry.json`
- `setup_summary.json`
- `source_fingerprints.json`
- `frozen_fold_manifest.parquet`

### Required Setup Conditions

The following upstream conditions must be satisfied:

1. official data paths were resolved successfully;
2. the feature-label schema was validated;
3. the frozen grouped folds were recovered;
4. the transcript source was identified;
5. the previous setup approved the start of the turn-parser phase;
6. all registry paths required by this notebook are present and absolute;
7. the registered project and setup paths agree with the current repository;
8. the frozen-fold manifest is readable and structurally valid.

### Execution Policy

This is a blocking gate.

- `PASS` — continue to official source verification;
- `WARNING` — continue only when the issue is non-blocking and explicitly documented;
- `FAIL` — stop the notebook before inspecting or publishing downstream data artefacts.

Paths are loaded from the upstream registry rather than being manually duplicated in this notebook.

In [3]:
SETUP_RELATIVE_DIR = Path("scratch_mastery_outputs") / "00_project_setup"

SETUP_ARTIFACT_NAMES = {
    "path_registry": "path_registry.json",
    "setup_summary": "setup_summary.json",
    "source_fingerprints": "source_fingerprints.json",
    "frozen_fold_manifest": "frozen_fold_manifest.parquet"
}


def locate_project_root(start_path, marker_relative_path):
    start_path = Path(start_path).resolve()
    candidate_roots = [start_path, *start_path.parents]
    matching_roots = [root for root in candidate_roots if (root / marker_relative_path).is_file()]

    if not matching_roots:
        searched_paths = [str(root / marker_relative_path) for root in candidate_roots]
        searched_text = "\n".join(f"- {path}" for path in searched_paths)

        raise FileNotFoundError(
            "Could not locate the project root from the current working directory.\n"
            f"Required marker: {marker_relative_path}\n"
            f"Searched:\n{searched_text}"
        )

    return matching_roots[0]


PROJECT_ROOT_DISCOVERED = locate_project_root(
    Path.cwd(),
    SETUP_RELATIVE_DIR / SETUP_ARTIFACT_NAMES["path_registry"]
)

SETUP_OUTPUT_DIR_DISCOVERED = PROJECT_ROOT_DISCOVERED / SETUP_RELATIVE_DIR

setup_artifact_paths = {
    name: SETUP_OUTPUT_DIR_DISCOVERED / file_name
    for name, file_name in SETUP_ARTIFACT_NAMES.items()
}

missing_setup_artifacts = [
    f"{name}: {path}"
    for name, path in setup_artifact_paths.items()
    if not path.is_file()
]

if missing_setup_artifacts:
    missing_text = "\n".join(f"- {item}" for item in missing_setup_artifacts)
    raise FileNotFoundError(f"Required setup artefacts are missing:\n{missing_text}")

path_registry = read_json(setup_artifact_paths["path_registry"])
setup_summary = read_json(setup_artifact_paths["setup_summary"])
source_fingerprints = read_json(setup_artifact_paths["source_fingerprints"])
frozen_fold_manifest = safe_table_read(setup_artifact_paths["frozen_fold_manifest"])

if not isinstance(path_registry, dict):
    raise TypeError("path_registry.json must contain one JSON object.")

if not isinstance(setup_summary, dict):
    raise TypeError("setup_summary.json must contain one JSON object.")

if not isinstance(source_fingerprints, dict):
    raise TypeError("source_fingerprints.json must contain one JSON object.")

if not isinstance(frozen_fold_manifest, pd.DataFrame):
    raise TypeError("frozen_fold_manifest.parquet did not load as a pandas DataFrame.")

setup_artifact_inventory = []

for artifact_name, artifact_path in setup_artifact_paths.items():
    status = path_status(artifact_path)

    setup_artifact_inventory.append({
        "artifact": artifact_name,
        "path": str(artifact_path),
        "exists": status["exists"],
        "readable": status["accessible"],
        "size_bytes": status["size_bytes"],
        "loaded_type": type({
            "path_registry": path_registry,
            "setup_summary": setup_summary,
            "source_fingerprints": source_fingerprints,
            "frozen_fold_manifest": frozen_fold_manifest
        }[artifact_name]).__name__
    })

setup_artifact_inventory = pd.DataFrame(setup_artifact_inventory)

print(f"Discovered project root: {PROJECT_ROOT_DISCOVERED}")
print(f"Loaded frozen-fold rows: {len(frozen_fold_manifest):,}")
display(setup_artifact_inventory)

Discovered project root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
Loaded frozen-fold rows: 35,072


,artifact,path,exists,readable,size_bytes,loaded_type
0,path_registry,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\pat...,True,True,3515,dict
1,setup_summary,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\set...,True,True,888,dict
2,source_fingerprints,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\sou...,True,True,637,dict
3,frozen_fold_manifest,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\fro...,True,True,615016,DataFrame


In [5]:
def as_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (int, np.integer)) and value in {0, 1}:
        return bool(value)
    if isinstance(value, str):
        value = value.strip().lower()
        if value in {"true", "yes", "1", "pass", "ready"}:
            return True
        if value in {"false", "no", "0", "fail", "not_ready"}:
            return False
    return None


def same_path(left, right):
    if left is None or right is None:
        return False
    left = os.path.normcase(os.path.normpath(str(Path(left).expanduser().resolve())))
    right = os.path.normcase(os.path.normpath(str(Path(right).expanduser().resolve())))
    return left == right


checks = []


def add_check(requirement, observed, expected, passed, blocking=True, detail=""):
    status = "PASS" if passed else ("FAIL" if blocking else "WARNING")
    checks.append({"requirement": requirement, "observed": observed, "expected": expected, "blocking": blocking, "status": status, "detail": detail})


required_flags = ["data_paths_ready", "feature_label_schema_ready", "frozen_folds_ready", "transcript_source_ready", "turn_parser_ready_to_start"]

for flag in required_flags:
    value = setup_summary.get(flag)
    add_check(f"Setup flag: {flag}", value, True, as_bool(value) is True)


transcript_source_type = path_registry.get("transcript_source_type")
required_registry_keys = ["project_root", "data_root", "scratch_output_root", "setup_output_dir", "train_features_path", "train_labels_path", "submission_format_path_1", "submission_format_path_2", "frozen_fold_manifest_path", "transcript_source_type"]

if transcript_source_type == "external_path":
    required_registry_keys.append("transcript_source")
elif transcript_source_type == "embedded_column":
    required_registry_keys.append("embedded_transcript_column")
else:
    add_check("Transcript source type", transcript_source_type, "external_path or embedded_column", False)

for key in required_registry_keys:
    value = path_registry.get(key)
    present = value is not None and str(value).strip() != ""
    add_check(f"Registry key: {key}", value, "Present and non-empty", present)


path_rules = {"project_root": "directory", "data_root": "directory", "scratch_output_root": "directory", "setup_output_dir": "directory", "train_features_path": "file", "train_labels_path": "file", "submission_format_path_1": "file", "submission_format_path_2": "file", "frozen_fold_manifest_path": "file"}

if transcript_source_type == "external_path":
    path_rules["transcript_source"] = "directory"

for key, expected_type in path_rules.items():
    value = path_registry.get(key)
    path = Path(value).expanduser() if value else None
    is_absolute = path is not None and path.is_absolute()
    type_valid = path is not None and (path.is_file() if expected_type == "file" else path.is_dir())

    add_check(f"Absolute path: {key}", value, "Absolute path", is_absolute)
    add_check(f"Path type: {key}", expected_type if type_valid else "Invalid", expected_type, type_valid)


path_matches = {"Project root consistency": (path_registry.get("project_root"), PROJECT_ROOT_DISCOVERED), "Setup directory consistency": (path_registry.get("setup_output_dir"), SETUP_OUTPUT_DIR_DISCOVERED), "Frozen-fold path consistency": (path_registry.get("frozen_fold_manifest_path"), setup_artifact_paths["frozen_fold_manifest"])}

for requirement, values in path_matches.items():
    observed, expected = values
    add_check(requirement, observed, str(expected), same_path(observed, expected))


required_fingerprint_keys = ["train_features_path", "train_features_sha256", "train_labels_path", "train_labels_sha256"]

for key in required_fingerprint_keys:
    value = source_fingerprints.get(key)
    present = value is not None and str(value).strip() != ""
    add_check(f"Fingerprint key: {key}", value, "Present and non-empty", present)


required_fold_columns = {"response_id", "session_id", "fold"}
available_fold_columns = set(frozen_fold_manifest.columns)
missing_fold_columns = sorted(required_fold_columns - available_fold_columns)

add_check("Frozen-fold manifest is non-empty", len(frozen_fold_manifest), "More than 0 rows", len(frozen_fold_manifest) > 0)
add_check("Frozen-fold required columns", sorted(available_fold_columns), sorted(required_fold_columns), not missing_fold_columns, detail=f"Missing columns: {missing_fold_columns}" if missing_fold_columns else "")

if not missing_fold_columns:
    fold_core = frozen_fold_manifest[["response_id", "session_id", "fold"]].copy()
    fold_values = pd.to_numeric(fold_core["fold"], errors="coerce")
    observed_folds = set(fold_values.dropna().astype(int).unique())

    null_count = int(fold_core.isna().sum().sum())
    duplicate_responses = int(fold_core["response_id"].duplicated().sum())
    session_overlap_count = int((fold_core.groupby("session_id")["fold"].nunique() > 1).sum())

    add_check("Frozen-fold required-field nulls", null_count, 0, null_count == 0)
    add_check("Duplicate response fold assignments", duplicate_responses, 0, duplicate_responses == 0)
    add_check("Frozen-fold domain", sorted(observed_folds), [0, 1, 2, 3, 4], observed_folds == {0, 1, 2, 3, 4})
    add_check("Sessions assigned to multiple folds", session_overlap_count, 0, session_overlap_count == 0)


git_safe = as_bool(setup_summary.get("scratch_output_git_safe"))
add_check("Scratch outputs ignored by Git", setup_summary.get("scratch_output_git_safe"), True, git_safe is True, blocking=False, detail="Non-blocking for source inspection, but required before publishing generated artefacts.")


setup_entry_gate = pd.DataFrame(checks)
blocking_failures = setup_entry_gate[setup_entry_gate["blocking"] & setup_entry_gate["status"].eq("FAIL")].copy()
SETUP_ENTRY_GATE_PASSED = blocking_failures.empty

entry_gate_summary = {"total_checks": int(len(setup_entry_gate)), "passed": int(setup_entry_gate["status"].eq("PASS").sum()), "warnings": int(setup_entry_gate["status"].eq("WARNING").sum()), "failed": int(setup_entry_gate["status"].eq("FAIL").sum()), "blocking_failures": int(len(blocking_failures)), "setup_entry_gate_passed": SETUP_ENTRY_GATE_PASSED}

display(setup_entry_gate)
display(pd.DataFrame([entry_gate_summary]))

if not SETUP_ENTRY_GATE_PASSED:
    failed_requirements = "\n".join(f"- {item}" for item in blocking_failures["requirement"])
    raise RuntimeError(f"SETUP_ENTRY_GATE_PASSED = False\nBlocking checks:\n{failed_requirements}")


PROJECT_ROOT = Path(path_registry["project_root"])
DATA_ROOT = Path(path_registry["data_root"])
SCRATCH_OUTPUT_ROOT = Path(path_registry["scratch_output_root"])
SETUP_OUTPUT_DIR = Path(path_registry["setup_output_dir"])

TRAIN_FEATURES_PATH = Path(path_registry["train_features_path"])
TRAIN_LABELS_PATH = Path(path_registry["train_labels_path"])
SUBMISSION_FORMAT_PATH_1 = Path(path_registry["submission_format_path_1"])
SUBMISSION_FORMAT_PATH_2 = Path(path_registry["submission_format_path_2"])
FROZEN_FOLD_MANIFEST_PATH = Path(path_registry["frozen_fold_manifest_path"])

TRANSCRIPT_SOURCE_TYPE = transcript_source_type
TRANSCRIPT_SOURCE_PATH = Path(path_registry["transcript_source"]) if transcript_source_type == "external_path" else None
EMBEDDED_TRANSCRIPT_COLUMN = path_registry.get("embedded_transcript_column")

INVENTORY_OUTPUT_DIR = SCRATCH_OUTPUT_ROOT / "01_data_foundation" / "01_inventory"
INVENTORY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_SOURCE_PATHS = {"train_features": TRAIN_FEATURES_PATH, "train_labels": TRAIN_LABELS_PATH, "submission_format_1": SUBMISSION_FORMAT_PATH_1, "submission_format_2": SUBMISSION_FORMAT_PATH_2, "frozen_fold_manifest": FROZEN_FOLD_MANIFEST_PATH}

if TRANSCRIPT_SOURCE_PATH is not None:
    OFFICIAL_SOURCE_PATHS["train_transcripts"] = TRANSCRIPT_SOURCE_PATH

print("SETUP_ENTRY_GATE_PASSED = True")
print(f"Inventory output directory: {INVENTORY_OUTPUT_DIR}")

,requirement,observed,expected,blocking,status,detail
0,Setup flag: data_paths_ready,True,True,True,PASS,
1,Setup flag: feature_label_schema_ready,True,True,True,PASS,
2,Setup flag: frozen_folds_ready,True,True,True,PASS,
3,Setup flag: transcript_source_ready,True,True,True,PASS,
4,Setup flag: turn_parser_ready_to_start,True,True,True,PASS,
5,Registry key: project_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition,Present and non-empty,True,PASS,
6,Registry key: data_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset,Present and non-empty,True,PASS,
7,Registry key: scratch_output_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs,Present and non-empty,True,PASS,
8,Registry key: setup_output_dir,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup,Present and non-empty,True,PASS,
9,Registry key: train_features_path,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_features_TMQTWs...,Present and non-empty,True,PASS,


,total_checks,passed,warnings,failed,blocking_failures,setup_entry_gate_passed
0,50,50,0,0,0,True


SETUP_ENTRY_GATE_PASSED = True
Inventory output directory: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\01_data_foundation\01_inventory
